# 00 — Optionally define `mask_pixel` with Paint

This is the PNG/Paint alternative to `00a_define_mask_pixel_napari.ipynb`. It loads raw detector images and creates one raw-coordinate mask per image **before FTH**. Run it before optional stitching as well, because stitching uses these masks.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np

def find_project_root(start=None):
    folder = Path(start or Path.cwd()).resolve()
    for candidate in (folder, *folder.parents):
        if (candidate / "library").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root containing library/.")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))
from library.data_loading import SextantsNexusLoader
from library.mask_store import MaskStore
from library.painted_masks import mask_png_paths, save_mask_reference_png
print("Project root:", ROOT)

## Select raw images and export Paint references

In [ ]:
# Include every raw image used directly by FTH or by stitching.
IMAGE_IDS = [589, 590]
RAW_DATA_FOLDER = ROOT / "SOLEIL_2507" / "data" / "20250723"
loader = SextantsNexusLoader(RAW_DATA_FOLDER)
mask_store = MaskStore(ROOT / "processed" / "mask_pixels")

frames = {image_id: loader.load(image_id) for image_id in IMAGE_IDS}
for image_id, frame in frames.items():
    reference_png, painted_png = mask_png_paths(ROOT, "mask_pixel", image_id)
    save_mask_reference_png(frame.image, reference_png, log_scale=True)
    print(f"Image {image_id}: {frame.image.shape}, exposure={frame.exposure:g}")
    print("  Open in Paint:", reference_png)
    print("  Paint mask pixels bright red and save as:", painted_png)

## Paint and save

Open each `_reference.png`, paint unusable pixels pure red (RGB 255, 0, 0), and save it under the printed filename without `_reference`. Keep the canvas dimensions unchanged. Then run the cell below.

In [ ]:
fig, axes = plt.subplots(len(IMAGE_IDS), 2, figsize=(10, 4 * len(IMAGE_IDS)), squeeze=False)
for row, image_id in enumerate(IMAGE_IDS):
    frame = frames[image_id]
    mask_pixel = mask_store.load(image_id, frame.image.shape)
    # Re-save in canonical pure-red form after validation.
    saved_path = mask_store.save(image_id, mask_pixel)
    axes[row, 0].imshow(frame.image, cmap="gray")
    axes[row, 0].imshow(mask_pixel, alpha=0.35, cmap="Reds", vmin=0, vmax=1)
    axes[row, 0].set_title(f"raw image {image_id} + mask")
    axes[row, 1].imshow(mask_pixel, cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title(f"mask_pixel_{image_id}.png")
    for axis in axes[row]:
        axis.set_axis_off()
    print(f"Image {image_id}: saved {int(mask_pixel.sum())} masked pixels to {saved_path}")
fig.tight_layout()
plt.show()